In [2]:
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# Cleaned Dataset

In [3]:
df_train = pd.read_csv('../data/cleaned_train.csv') #train_lemmastop.csv
print(f"Shape: {df_train.shape}")
df_train.head()

Shape: (6827, 9)


,id,text,anger,fear,joy,sadness,surprise,emotions,clean_text
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness'],the dentist that did the work apparently did a...
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness'],i am going to absolutely be terrible during my...
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness'],bridge so leave me drowning calling houston an...
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness'],after that mess i went to see my now exgirlfri...
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear'],as he stumbled i ran off afraid it might someh...


In [4]:
df_test = pd.read_csv('../data/cleaned_test.csv') #test_lemmastop
print(f"Shape: {df_test.shape}")
df_test.head()

Shape: (1707, 3)


,id,text,clean_text
0,0,she wanted to fight over every single little t...,she wanted to fight over every single little t...
1,1,"anyway, back to tuesday.",anyway back to tuesday
2,2,she shrieked at the dog to go back.,she shrieked at the dog to go back
3,3,yelling for everyone to get back or get inside...,yelling for everyone to get back or get inside...
4,4,still kind of freaky.,still kind of freaky


## Naive Bayes (Multinomial NB)

In [5]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']

X = df_train['clean_text']
y = df_train[emotion_cols]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

Training set size: 5461
Validation set size: 1366


In [6]:
vectorizer = TfidfVectorizer(max_features=5000) # Can use Count Vectorizer also
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [7]:
nb_model = OneVsRestClassifier(MultinomialNB())
nb_model.fit(X_train_tfidf, y_train)

,estimator,MultinomialNB()
,n_jobs,None
,verbose,0
,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [8]:
y_pred_nb = nb_model.predict(X_val_tfidf)
f1_nb = f1_score(y_val, y_pred_nb, average='macro', zero_division=0)
f1_nb

0.2982695052454626

### For Test Dataset

In [9]:
X_test = df_test['clean_text']
X_tfidf = vectorizer.fit_transform(X)
X_test_tfidf = vectorizer.transform(X_test)

In [10]:
nb_model = OneVsRestClassifier(MultinomialNB())
nb_model.fit(X_tfidf, y)

,estimator,MultinomialNB()
,n_jobs,None
,verbose,0
,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [11]:
y_test_pred = nb_model.predict(X_test_tfidf)
sub = pd.DataFrame(y_test_pred, columns=['anger', 'fear', 'joy', 'sadness', 'surprise'])

sub['id'] = range(len(y_test_pred))
sub = sub[['id', 'anger', 'fear', 'joy', 'sadness', 'surprise']]
sub.head()
# sub.to_csv('submission.csv', index=False)

,id,anger,fear,joy,sadness,surprise
0,0,0,1,0,0,0
1,1,0,0,0,0,0
2,2,0,1,0,0,0
3,3,0,1,0,0,0
4,4,0,1,0,0,1


## SVM

In [12]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']

X = df_train['clean_text']
y = df_train[emotion_cols]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

Training set size: 5461
Validation set size: 1366


In [13]:
vectorizer = TfidfVectorizer(max_features=5000) # Can use Count Vectorizer also
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [14]:
svm_model = OneVsRestClassifier(LinearSVC(random_state=42))
svm_model.fit(X_train_tfidf, y_train)

,estimator,LinearSVC(random_state=42)
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [15]:
y_pred_svm = svm_model.predict(X_val_tfidf)
f1_svm = f1_score(y_val, y_pred_svm, average='macro', zero_division=0)
f1_svm

0.6759554429377237

### For Test Dataset

In [16]:
X_test = df_test['clean_text']
X_tfidf = vectorizer.fit_transform(X)
X_test_tfidf = vectorizer.transform(X_test)

In [17]:
svm_model = OneVsRestClassifier(LinearSVC(random_state=42))
svm_model.fit(X_tfidf, y)

,estimator,LinearSVC(random_state=42)
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [18]:
y_test_pred = svm_model.predict(X_test_tfidf)
sub = pd.DataFrame(y_test_pred, columns=['anger', 'fear', 'joy', 'sadness', 'surprise'])

sub['id'] = range(len(y_test_pred))
sub = sub[['id', 'anger', 'fear', 'joy', 'sadness', 'surprise']]
sub.head()
# sub.to_csv('submission.csv', index=False)

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,0,1
1,1,0,0,0,0,0
2,2,1,1,0,0,0
3,3,0,1,0,0,0
4,4,0,1,0,0,1


# LemmaStop Dataset

In [19]:
df_train = pd.read_csv('../data/train_lemmastop.csv')
print(f"Shape: {df_train.shape}")
df_train.head()

Shape: (6827, 10)


,id,text,anger,fear,joy,sadness,surprise,emotions,clean_text,processed_text
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness'],the dentist that did the work apparently did a...,dentist work apparently lousy job year teeth d...
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness'],i am going to absolutely be terrible during my...,going absolutely terrible first sexual experience
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness'],bridge so leave me drowning calling houston an...,bridge leave drowning calling houston let lung...
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness'],after that mess i went to see my now exgirlfri...,mess went see exgirlfriend school refused driv...
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear'],as he stumbled i ran off afraid it might someh...,stumbled ran afraid might somehow affect job s...


In [20]:
df_test = pd.read_csv('../data/test_lemmastop.csv')
print(f"Shape: {df_test.shape}")
df_test.head()

Shape: (1707, 4)


,id,text,clean_text,processed_text
0,0,she wanted to fight over every single little t...,she wanted to fight over every single little t...,wanted fight every single little thing
1,1,"anyway, back to tuesday.",anyway back to tuesday,anyway back tuesday
2,2,she shrieked at the dog to go back.,she shrieked at the dog to go back,shrieked dog go back
3,3,yelling for everyone to get back or get inside...,yelling for everyone to get back or get inside...,yelling everyone get back get inside draw knif...
4,4,still kind of freaky.,still kind of freaky,still kind freaky


## Naive Bayes (Multinomial NB)

In [21]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']
df_train['processed_text'] = df_train['processed_text'].fillna('')

X = df_train['processed_text']
y = df_train[emotion_cols]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

Training set size: 5461
Validation set size: 1366


In [22]:
vectorizer = TfidfVectorizer(max_features=5000) # Can use Count Vectorizer also
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [23]:
nb_model = OneVsRestClassifier(MultinomialNB())
nb_model.fit(X_train_tfidf, y_train)

,estimator,MultinomialNB()
,n_jobs,None
,verbose,0
,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [24]:
y_pred_nb = nb_model.predict(X_val_tfidf)
f1_nb = f1_score(y_val, y_pred_nb, average='macro', zero_division=0)
f1_nb

0.36233510622147647

### For Test Dataset

In [25]:
df_test['processed_text'] = df_test['processed_text'].fillna('')
X_test = df_test['processed_text']
X_tfidf = vectorizer.fit_transform(X)
X_test_tfidf = vectorizer.transform(X_test)

In [26]:
nb_model = OneVsRestClassifier(MultinomialNB())
nb_model.fit(X_tfidf, y)

,estimator,MultinomialNB()
,n_jobs,None
,verbose,0
,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [27]:
y_test_pred = nb_model.predict(X_test_tfidf)
sub = pd.DataFrame(y_test_pred, columns=['anger', 'fear', 'joy', 'sadness', 'surprise'])

sub['id'] = range(len(y_test_pred))
sub = sub[['id', 'anger', 'fear', 'joy', 'sadness', 'surprise']]
sub.head()
# sub.to_csv('submission.csv', index=False)

,id,anger,fear,joy,sadness,surprise
0,0,0,1,0,0,0
1,1,0,0,0,0,0
2,2,0,1,0,0,0
3,3,0,1,0,0,0
4,4,0,1,0,0,1


## SVM

In [28]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']
df_train['processed_text'] = df_train['processed_text'].fillna('')

X = df_train['processed_text']
y = df_train[emotion_cols]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

Training set size: 5461
Validation set size: 1366


In [29]:
vectorizer = TfidfVectorizer(max_features=5000) # Can use Count Vectorizer also
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [30]:
svm_model = OneVsRestClassifier(LinearSVC(random_state=42))
svm_model.fit(X_train_tfidf, y_train)

,estimator,LinearSVC(random_state=42)
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [31]:
y_pred_svm = svm_model.predict(X_val_tfidf)
f1_svm = f1_score(y_val, y_pred_svm, average='macro', zero_division=0)
f1_svm

0.6877940869024731

### For Test Dataset

In [32]:
df_test['processed_text'] = df_test['processed_text'].fillna('')
X_test = df_test['processed_text']
X_tfidf = vectorizer.fit_transform(X)
X_test_tfidf = vectorizer.transform(X_test)

In [33]:
svm_model = OneVsRestClassifier(LinearSVC(random_state=42))
svm_model.fit(X_tfidf, y)

,estimator,LinearSVC(random_state=42)
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [34]:
y_test_pred = svm_model.predict(X_test_tfidf)
sub = pd.DataFrame(y_test_pred, columns=['anger', 'fear', 'joy', 'sadness', 'surprise'])

sub['id'] = range(len(y_test_pred))
sub = sub[['id', 'anger', 'fear', 'joy', 'sadness', 'surprise']]
sub.head()
# sub.to_csv('submission.csv', index=False)

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,0,0
1,1,0,0,0,0,0
2,2,1,1,0,0,0
3,3,0,1,0,0,0
4,4,0,1,0,0,1
